# **04 · Unificación de los datasets**

Este notebook junta los tres datasets procesados en una sola tabla, les agrega los feriados de su país y marca qué filas son de entrenamiento, validación y prueba. El resultado es la entrada del entrenamiento.

## **0. Objetivo**

- **Formato común:** los tres datasets ya tienen las mismas columnas (`dataset`, `store_id`, `product_id`, `date`, `sales`, `is_open`). Aquí se verifica y se juntan.
- **Feriados:** se agregan `is_holiday` e `is_holiday_next2days` con el calendario de cada país, con la misma lógica para los tres datasets.
- **División:** cada fila queda marcada como entrenamiento, validación o prueba, de forma que el modelo nunca vea datos futuros ni negocios de la prueba.

## **1. Carga y verificación**

In [1]:
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"

files = {
    "ddop": "ddop_bakery.parquet",
    "french_bakery": "french_bakery.parquet",
    "bread_basket": "bread_basket.parquet",
}
parts = {name: pd.read_parquet(PROCESSED / file) for name, file in files.items()}

pd.DataFrame({name: part.dtypes.astype(str) for name, part in parts.items()})

,ddop,french_bakery,bread_basket
dataset,str,str,str
store_id,str,str,str
product_id,str,str,str
date,datetime64[us],datetime64[us],datetime64[us]
sales,int64,int64,int64
is_open,bool,bool,bool


In [2]:
reference = parts["ddop"].dtypes
assert all(part.dtypes.equals(reference) for part in parts.values()), "Los datasets no tienen las mismas columnas y tipos"

df = pd.concat(parts.values(), ignore_index=True)
summary = df.groupby("dataset").agg(
    filas=("sales", "size"),
    locales=("store_id", "nunique"),
    productos=("product_id", "nunique"),
    desde=("date", "min"),
    hasta=("date", "max"),
)
summary["series"] = df.groupby("dataset").apply(lambda g: g.groupby(["store_id", "product_id"]).ngroups)
summary

,filas,locales,productos,desde,hasta,series
dataset,,,,,,
bread_basket,3144,1,20,2016-10-30,2017-04-09,20
ddop,127575,35,3,2016-01-02,2019-04-30,105
french_bakery,27953,1,47,2021-01-02,2022-09-30,47


## **2. Feriados**

Los feriados se construyen con la librería `holidays`, que tiene los calendarios oficiales de cada país. Cada dataset usa el calendario de su ubicación:

- **ddop:** Alemania. El dataset original indica feriados nacionales; en la sección 3 se verifica qué calendario coincide mejor.
- **French Bakery:** Francia.
- **The Bread Basket:** Escocia, porque Edimburgo tiene feriados propios que no aplican al resto de Reino Unido (por ejemplo, el 2 de enero).

Se crean dos variables:

- `is_holiday`: el día es feriado.
- `is_holiday_next2days`: mañana o pasado mañana es feriado, la misma definición que usa ddop.

In [3]:
def holiday_dates(country, years, subdiv=None):
    calendar = holidays.country_holidays(country, subdiv=subdiv, years=years)
    return pd.to_datetime(sorted(calendar.keys()))


def add_holidays(part, country, subdiv=None):
    years = range(part["date"].dt.year.min(), part["date"].dt.year.max() + 2)
    dates = holiday_dates(country, years, subdiv)
    next_two = dates - pd.Timedelta(days=1)
    next_two = next_two.union(dates - pd.Timedelta(days=2))
    return part.assign(
        is_holiday=part["date"].isin(dates),
        is_holiday_next2days=part["date"].isin(next_two),
    )

## **3. Validación de los feriados**

ddop trae su propia columna de feriados, así que se puede comprobar si la librería coincide. Se compara el calendario nacional de Alemania y el de algunos estados, porque en Alemania varios feriados son solo regionales.

In [4]:
raw_ddop = pd.read_csv(ROOT / "data" / "raw" / "ddop" / "bakery_data.csv", parse_dates=["date"])
ddop_holidays = set(raw_ddop.loc[raw_ddop["is_holiday"] == 1, "date"])
years = range(2016, 2020)

comparison = {}
for label, subdiv in [("Nacional", None), ("Baden-Württemberg", "BW"), ("Baviera", "BY"), ("Renania del Norte-Westfalia", "NW"), ("Baja Sajonia", "NI")]:
    lib = set(holiday_dates("DE", years, subdiv))
    lib = {d for d in lib if raw_ddop["date"].min() <= d <= raw_ddop["date"].max()}
    comparison[label] = {
        "en ambos": len(lib & ddop_holidays),
        "solo en ddop": len(ddop_holidays - lib),
        "solo en la librería": len(lib - ddop_holidays),
    }
pd.DataFrame(comparison).T

,en ambos,solo en ddop,solo en la librería
Nacional,30,17,0
Baden-Württemberg,40,7,0
Baviera,40,7,0
Renania del Norte-Westfalia,36,11,0
Baja Sajonia,30,17,1


In [5]:
national = {d for d in holiday_dates("DE", years) if raw_ddop["date"].min() <= d <= raw_ddop["date"].max()}
only_ddop = sorted(ddop_holidays - national)
pd.DataFrame({"fecha": only_ddop, "día": [d.day_name() for d in only_ddop]}).head(20)

,fecha,día
0,2016-01-06,Wednesday
1,2016-03-27,Sunday
2,2016-05-15,Sunday
3,2016-05-26,Thursday
4,2016-11-01,Tuesday
5,2017-01-06,Friday
6,2017-04-16,Sunday
7,2017-06-04,Sunday
8,2017-06-15,Thursday
9,2017-11-01,Wednesday


**Conclusión de la validación:** ddop no usa solo los feriados nacionales, aunque su descripción lo dice: incluye feriados regionales del sur de Alemania, como Reyes (6 de enero), Corpus Christi y Todos los Santos. Con el calendario de Baden-Württemberg coinciden 40 de los 47 feriados (Baviera da el mismo resultado en este periodo) y la librería no marca ninguno que ddop no tenga. Los 7 restantes son el Domingo de Pascua y el Domingo de Pentecostés, que ddop marca aunque no son feriados oficiales.

Se usa el **calendario regional de Baden-Württemberg** en lugar de la columna original, para que los tres datasets se construyan con el mismo método: el calendario oficial de la región del negocio, que es como funcionaría en Restocker para cualquier negocio nuevo.

In [6]:
calendars = {
    "ddop": ("DE", "BW"),            # Baden-Württemberg
    "french_bakery": ("FR", None),   # Francia
    "bread_basket": ("GB", "SCT"),   # Escocia
}
df = pd.concat(
    [add_holidays(df[df["dataset"] == name], country, subdiv) for name, (country, subdiv) in calendars.items()],
    ignore_index=True,
)

days = df.drop_duplicates(["dataset", "date"])
days.groupby("dataset")[["is_holiday", "is_holiday_next2days"]].sum().rename(columns={
    "is_holiday": "días feriados", "is_holiday_next2days": "días previos a un feriado"
})

,días feriados,días previos a un feriado
dataset,,
bread_basket,7,10
ddop,40,78
french_bakery,18,36


In [7]:
bb_closed = df[(df["dataset"] == "bread_basket") & ~df["is_open"]].drop_duplicates("date")[["date", "is_holiday"]]
bb_closed.assign(día=bb_closed["date"].dt.day_name())

,date,is_holiday,día
155582,2016-12-25,True,Sunday
155583,2016-12-26,True,Monday
155590,2017-01-02,True,Monday


## **4. División de los datos**

Cada fila se marca con una de cuatro partes, respetando siempre el orden en el tiempo para que el modelo nunca vea datos posteriores a los que predice:

- **Prueba (`test`):** 5 locales completos de ddop elegidos al azar con una semilla fija (cerca del 15 % de los 35) y todo The Bread Basket. Simula negocios que el modelo nunca vio y es la prueba principal.
- **Prueba de temporada (`test_season`):** julio a septiembre de 2022 de French Bakery, en pleno pico turístico. Mide cuánto sufre el modelo con un cambio brusco de nivel en un negocio que sí conoce; se reporta por separado.
- **Validación (`val`):** los últimos 3 meses de ddop (del 31 de enero al 30 de abril de 2019) y abril a junio de 2022 de French Bakery. La de French Bakery se coloca antes del verano para que la validación represente meses normales y no solo el pico turístico, que la prueba principal casi no tiene.
- **Entrenamiento (`train`):** el resto de ddop y French Bakery hasta antes de la validación, incluido el verano de 2021.

La validación sirve para detener el entrenamiento a tiempo y comparar iteraciones. Las dos pruebas se usan una sola vez, al final.

In [8]:
SEED = 123
ddop_stores = sorted(df.loc[df["dataset"] == "ddop", "store_id"].unique())
test_stores = sorted(str(s) for s in np.random.default_rng(SEED).choice(ddop_stores, size=5, replace=False))
print("Locales de ddop reservados para la prueba:", test_stores)

ddop_end = df.loc[df["dataset"] == "ddop", "date"].max()
periods = {
    "ddop": {"val": ddop_end - pd.DateOffset(months=3) + pd.Timedelta(days=1)},
    "french_bakery": {"val": pd.Timestamp("2022-04-01"), "test_season": pd.Timestamp("2022-07-01")},
}
print("Periodos:", {name: {k: str(v.date()) for k, v in p.items()} for name, p in periods.items()})

df["split"] = "train"
is_ddop, is_fb = df["dataset"] == "ddop", df["dataset"] == "french_bakery"
df.loc[is_ddop & (df["date"] >= periods["ddop"]["val"]), "split"] = "val"
df.loc[is_fb & (df["date"] >= periods["french_bakery"]["val"]), "split"] = "val"
df.loc[is_fb & (df["date"] >= periods["french_bakery"]["test_season"]), "split"] = "test_season"
df.loc[df["store_id"].isin(test_stores) | (df["dataset"] == "bread_basket"), "split"] = "test"

Locales de ddop reservados para la prueba: ['ddop_17', 'ddop_19', 'ddop_37', 'ddop_4', 'ddop_5']
Periodos: {'ddop': {'val': '2019-01-31'}, 'french_bakery': {'val': '2022-04-01', 'test_season': '2022-07-01'}}


## **5. Resumen y exportación**

In [9]:
df["series"] = df["store_id"] + "/" + df["product_id"]
overview = df.groupby(["split", "dataset"]).agg(
    filas=("sales", "size"),
    series=("series", "nunique"),
    desde=("date", "min"),
    hasta=("date", "max"),
)
overview.loc[["train", "val", "test", "test_season"]]

filas  series      desde      hasta
split       dataset                                            
train       ddop           101250      90 2016-01-02 2019-01-30
            french_bakery   19749      47 2021-01-02 2022-03-31
val         ddop             8100      90 2019-01-31 2019-04-30
            french_bakery    4154      46 2022-04-01 2022-06-30
test        bread_basket     3144      20 2016-10-30 2017-04-09
            ddop            18225      15 2016-01-02 2019-04-30
test_season french_bakery    4050      45 2022-07-01 2022-09-30

In [10]:
share = df.groupby("split").size() / len(df)
print("Proporción de filas:", {k: f"{v:.1%}" for k, v in share.loc[["train", "val", "test", "test_season"]].items()})

OUT = PROCESSED / "unified.parquet"
df.drop(columns="series").to_parquet(OUT, index=False)
print(f"Guardado en {OUT.relative_to(ROOT)} ({OUT.stat().st_size / 1024:,.0f} KB) | Columnas: {list(df.drop(columns='series').columns)}")

Proporción de filas: {'train': '76.3%', 'val': '7.7%', 'test': '13.5%', 'test_season': '2.6%'}
Guardado en data/processed/unified.parquet (429 KB) | Columnas: ['dataset', 'store_id', 'product_id', 'date', 'sales', 'is_open', 'is_holiday', 'is_holiday_next2days', 'split']


## **6. Estructura del dataset unificado**

`unified.parquet` es la entrada del entrenamiento. Tiene una fila por local, producto y día, con estas columnas:

- `dataset`: de qué fuente viene (`ddop`, `french_bakery` o `bread_basket`).
- `store_id`: identificador del local.
- `product_id`: identificador del producto.
- `date`: el día.
- `sales`: unidades vendidas. **Es la variable que se quiere predecir.**
- `is_open`: si el local abrió ese día. Los días cerrados no cuentan como demanda 0 al medir el error.
- `is_holiday`: si el día es feriado en la región del negocio.
- `is_holiday_next2days`: si mañana o pasado mañana es feriado.
- `split`: la parte a la que pertenece la fila (`train`, `val`, `test` o `test_season`).

`dataset`, `store_id` y `product_id` sirven para separar las series y dividir los datos; no entran al modelo como variables. El día de la semana tampoco se guarda, porque se calcula a partir de `date` cuando se necesite.

In [11]:
unified = pd.read_parquet(OUT)
unified.dtypes.astype(str).rename("tipo").to_frame()

,tipo
dataset,str
store_id,str
product_id,str
date,datetime64[us]
sales,int64
is_open,bool
is_holiday,bool
is_holiday_next2days,bool
split,str


In [12]:
unified.sample(8, random_state=SEED).sort_values(["dataset", "date"])

,dataset,store_id,product_id,date,sales,is_open,is_holiday,is_holiday_next2days,split
155860,bread_basket,bread_basket_1,bread_basket_brownie,2016-11-14,0,True,False,False,test
103413,ddop,ddop_47,ddop_109,2016-05-19,11,True,False,False,train
30622,ddop,ddop_26,ddop_109,2016-09-05,6,True,False,False,train
106000,ddop,ddop_48,ddop_101,2016-10-23,100,True,False,False,train
50221,ddop,ddop_30,ddop_110,2017-02-11,160,True,False,False,train
27181,ddop,ddop_25,ddop_109,2017-03-28,18,True,False,False,train
49509,ddop,ddop_30,ddop_109,2018-06-29,49,True,False,False,train
143148,french_bakery,french_bakery_1,french_bakery_milles_feuilles,2021-06-10,1,True,False,False,train


Una misma serie pasa por varias partes a lo largo del tiempo. Por ejemplo, la baguette tradicional de French Bakery es de entrenamiento hasta marzo de 2022, de validación de abril a junio y de prueba de temporada desde julio:

In [13]:
serie = unified[unified["product_id"] == "french_bakery_traditional_baguette"]
serie.groupby("split", sort=False).agg(desde=("date", "min"), hasta=("date", "max"), dias=("date", "size"), venta_promedio=("sales", "mean")).round({"venta_promedio": 1})

,desde,hasta,dias,venta_promedio
split,,,,
train,2021-01-02,2022-03-31,454,165.6
val,2022-04-01,2022-06-30,91,167.8
test_season,2022-07-01,2022-09-30,92,293.7


In [14]:
print(f"Filas: {len(unified):,} | Series: {unified.groupby(['store_id', 'product_id']).ngroups} | Valores nulos: {int(unified.isna().sum().sum())}")

Filas: 158,672 | Series: 172 | Valores nulos: 0
